# 03 – Propiedades Dependientes de la Temperatura y Gráficos

Una capacidad calorífica constante es una buena aproximación en una ventana
estrecha de temperatura, pero la combustión, las turbinas de gas y los flujos
reactivos abarcan cientos — incluso miles — de kelvin. En ese régimen, $C_p$,
$S^\circ$ y $H^\circ$ varían fuertemente, y `pyglenn` proporciona su verdadera
dependencia con la temperatura directamente desde los polinomios de la NASA.

En este cuaderno:

1. barremos la temperatura y construimos curvas de propiedades para varios
   gases;
2. leemos la física detrás de las curvas $C_p(T)$ (equipartición y vibraciones);
3. tabulamos los resultados con `pandas`;
4. contrastamos las capacidades caloríficas *instantánea* y *media*; y
5. vemos cómo las fases condensadas tienen rangos de validez más estrechos.

In [ ]:
from pyglenn import ThermochemicalCalculator, R

_INDEX = {}

def species_id(calc, name):
    """Devuelve el id en la base de datos de la especie cuyo *nombre* coincide exactamente.

    ``get_available_species`` busca subcadenas tanto en el nombre como en la
    fórmula y limita el resultado a 20 filas, por lo que nombres cortos como
    ``"O2"`` pueden quedar desplazados por entradas como ``"AL2O2"`` o
    ``"CO2"``. Para ser robustos, construimos un índice completo nombre -> id
    una sola vez (almacenado en caché durante la sesión) y buscamos el nombre
    exacto en él.
    """
    if not _INDEX:
        _INDEX.update({s["name"]: s["id"] for s in calc.get_available_species("")})
    if name not in _INDEX:
        raise ValueError(f"Especie {name!r} no encontrada en la base de datos")
    return _INDEX[name]

print("Constante universal de los gases R =", R, "J/(mol.K)")


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


## 1. Barriendo la temperatura

`get_properties_range(id, temps)` evalúa una lista completa de temperaturas de
una sola vez y devuelve un diccionario indexado por temperatura. Lo envolvemos
en un pequeño auxiliar que devuelve arrays listos para graficar, ignorando
cualquier temperatura que esté fuera del rango de validez de la especie (estas
simplemente no aparecen).

In [ ]:
SPECIES = ["Ar", "H2", "N2", "O2", "H2O", "CO2", "CH4"]

def curve(calc, name, temps):
    """Devuelve arrays (T, cp, s, h) para las temperaturas que están dentro del rango."""
    sid = species_id(calc, name)
    table = calc.get_properties_range(sid, list(temps)) or {}
    T = np.array(sorted(table))
    cp = np.array([table[t]["cp"] for t in T])
    s = np.array([table[t]["s"] for t in T])
    h = np.array([table[t]["h_relative"] for t in T])
    return T, cp, s, h

temps = np.linspace(300, 3000, 55)
data = {}
with ThermochemicalCalculator() as calc:
    for name in SPECIES:
        data[name] = curve(calc, name, temps)
print("Curvas calculadas para:", ", ".join(data))

## 2. Tres curvas de propiedades

$C_p^\circ(T)$, $S^\circ(T)$ y la **entalpía sensible**
$H^\circ(T)-H^\circ(298{,}15\,\mathrm{K})$ (la entalpía necesaria para calentar
la especie desde la temperatura ambiente). Restamos el valor a 298,15 K para que
cada curva comience cerca de cero y las formas sean comparables,
independientemente de la entalpía de formación.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
with ThermochemicalCalculator() as calc:
    h_ref = {n: calc.calculate_properties(species_id(calc, n), 298.15)["h_relative"]
             for n in SPECIES}

for name in SPECIES:
    T, cp, s, h = data[name]
    axes[0].plot(T, cp, label=name)
    axes[1].plot(T, s, label=name)
    axes[2].plot(T, (h - h_ref[name]) / 1000.0, label=name)

axes[0].set_title(r"Capacidad calorífica $C_p^\circ(T)$"); axes[0].set_ylabel("J/(mol.K)")
axes[1].set_title(r"Entropía absoluta $S^\circ(T)$"); axes[1].set_ylabel("J/(mol.K)")
axes[2].set_title("Entalpía sensible $H(T)-H(298)$"); axes[2].set_ylabel("kJ/mol")
for ax in axes:
    ax.set_xlabel("Temperatura [K]"); ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 3. Leyendo la física: equipartición

La capacidad calorífica adimensional $C_p/R$ revela la estructura molecular. Por
equipartición, cada grado de libertad cuadrático totalmente activo contribuye con
$\tfrac12 R$ a $C_v$, y para un gas ideal $C_p = C_v + R$:

* **Monoatómico** (Ar): solo 3 modos traslacionales $\Rightarrow C_v=\tfrac32
  R$, luego $C_p=\tfrac52 R$ — constante a todas las temperaturas.
* **Diatómico** (N₂, O₂, H₂): + 2 modos rotacionales $\Rightarrow
  C_p=\tfrac72 R$ a $T$ moderada, subiendo aún más cuando el modo vibracional se
  activa.
* **Poliatómico** (CO₂, H₂O, CH₄): más modos rotacionales y muchos modos
  vibracionales, por lo que $C_p/R$ es mayor y sube pronunciadamente.

Las líneas discontinuas marcan las mesetas $\tfrac52$ y $\tfrac72$.

In [ ]:
fig, ax = plt.subplots()
for name in SPECIES:
    T, cp, s, h = data[name]
    ax.plot(T, cp / R, label=name)
for level, txt in [(2.5, "5/2  (monoatómico)"), (3.5, "7/2  (diatómico, sin vib.)")]:
    ax.axhline(level, ls="--", color="0.7")
    ax.text(310, level + 0.05, txt, fontsize=9, color="0.4")
ax.set_xlabel("Temperatura [K]")
ax.set_ylabel(r"$C_p / R$")
ax.set_title("La capacidad calorífica revela los grados de libertad moleculares")
ax.legend(fontsize=8)
plt.show()

## 4. Una tabla de propiedades con pandas

Las tablas discretas siguen siendo útiles para informes. Aquí está $C_p$ en un
conjunto de temperaturas para cada especie, ordenado por el valor a 2000 K para
clasificarlas por cuánta energía absorben cuando se calientan.

In [ ]:
table_temps = [300, 600, 1000, 1500, 2000, 2500]
records = {}
with ThermochemicalCalculator() as calc:
    for name in SPECIES:
        sid = species_id(calc, name)
        records[name] = {f"{t} K": calc.calculate_properties(sid, t)["cp"]
                         for t in table_temps}

cp_df = pd.DataFrame(records).T.sort_values("2000 K", ascending=False)
cp_df.index.name = "especie"
print("Cp [J/(mol.K)]")
print(cp_df.to_string())

## 5. Capacidad calorífica instantánea vs. media

Dos capacidades caloríficas aparecen en cálculos de ingeniería:

* la **instantánea** $C_p(T) = \left(\partial H/\partial T\right)_p$;
* la **media** $\overline{C_p}\big|_{T_1}^{T_2} = \dfrac{H(T_2)-H(T_1)}{T_2-T_1}$,
  que es la que realmente se multiplica por $\Delta T$ para obtener un cambio de
  entalpía.

Usando `calculate_enthalpy_change` para el numerador, comparamos ambas para el
nitrógeno. La media (desde 300 K) va por debajo del valor instantáneo porque
promedia la región más fría, de menor $C_p$.

In [ ]:
T2 = np.linspace(400, 3000, 40)
with ThermochemicalCalculator() as calc:
    n2 = species_id(calc, "N2")
    cp_inst = np.array([calc.calculate_properties(n2, t)["cp"] for t in T2])
    cp_mean = np.array([calc.calculate_enthalpy_change(n2, 300.0, t) / (t - 300.0)
                        for t in T2])

fig, ax = plt.subplots()
ax.plot(T2, cp_inst, label=r"instantánea $C_p(T)$")
ax.plot(T2, cp_mean, "--", label=r"media $\overline{C_p}\,|_{300}^{T}$")
ax.set_xlabel("Temperatura [K]")
ax.set_ylabel("Capacidad calorífica del N$_2$ [J/(mol.K)]")
ax.set_title("Capacidad calorífica instantánea vs. media del nitrógeno")
ax.legend()
plt.show()

## 6. Las fases condensadas tienen rangos más estrechos

Los ajustes para fase gaseosa típicamente cubren 200–6000 K, pero los líquidos y
sólidos solo son válidos en una ventana pequeña. Solicitar una temperatura fuera
de esa ventana devuelve `None` — siempre maneje ese caso. El agua líquida es un
buen ejemplo.

In [ ]:
with ThermochemicalCalculator() as calc:
    liq = species_id(calc, "H2O(L)")
    data_liq = calc.db.get_species_data(liq)
    ranges = [(iv["temp_min"], iv["temp_max"]) for iv in data_liq["intervals"]]
    print("H2O(L) intervalo(s) válido(s):", ranges)
    for T in [350.0, 2000.0]:
        p = calc.calculate_properties(liq, T)
        status = "None (fuera de rango)" if p is None else f"Cp = {p['cp']:.2f} J/(mol.K)"
        print(f"  T = {T:7.1f} K  ->  {status}")

## Resumen

- `get_properties_range` barre muchas temperaturas en una sola llamada.
- La forma de $C_p/R$ refleja directamente los grados de libertad traslacionales,
  rotacionales y vibracionales.
- $C_p$ media e instantánea son diferentes — use la media (vía
  `calculate_enthalpy_change`) para $\Delta T$ finito.
- Siempre maneje `None` para temperaturas fuera de rango, especialmente en fases
  condensadas.

**A continuación:** el cuaderno 04 extrae entalpías de formación de estos mismos
polinomios.